# kaggle-vllm 0.2.0.dev0 — post-PR-15 full Kaggle T4×2 acceptance

This is an execution-ready replacement for the pending template. It pins the reviewed
post-PR-15 source commit `7327b0b0c811a92a9c49421a4d302c18e251ab61`, preserving the
T4/SM75 `TRITON_ATTN` profile where FlashInfer is optional.

**Kaggle settings:** Accelerator = GPU T4 ×2; Internet = ON.

## 1. Environment, exact source, and lightweight SDK build

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request

EXPECTED_SOURCE_COMMIT = "7327b0b0c811a92a9c49421a4d302c18e251ab61"
EXPECTED_SDK_VERSION = "0.2.0.dev0"

WORK = Path("/kaggle/working/kaggle-vllm-020-pr15")
SOURCE_ARCHIVE = WORK / f"kaggle-vllm-{EXPECTED_SOURCE_COMMIT}.tar.gz"
SOURCE_UNPACK = WORK / "source"
SDK_DIST = WORK / "dist"
SOURCE_URL = f"https://github.com/kaggle-vllm/kaggle-vllm/archive/{EXPECTED_SOURCE_COMMIT}.tar.gz"

def sha256_file(path: Path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_tar(archive: Path, destination: Path):
    destination = destination.resolve()
    with tarfile.open(archive, "r:gz") as tf:
        for member in tf.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing archive link member: {member.name}")
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tf.extractall(destination)

if WORK.exists():
    shutil.rmtree(WORK)
SDK_DIST.mkdir(parents=True, exist_ok=True)
SOURCE_UNPACK.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

torch_before = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda, torch.__file__)"],
    text=True,
)
print("Torch before SDK:", torch_before.strip())

print("Downloading exact reviewed source:", EXPECTED_SOURCE_COMMIT)
urllib.request.urlretrieve(SOURCE_URL, SOURCE_ARCHIVE)
safe_extract_tar(SOURCE_ARCHIVE, SOURCE_UNPACK)
roots = [p for p in SOURCE_UNPACK.iterdir() if p.is_dir()]
assert len(roots) == 1, roots
SOURCE_ROOT = roots[0]

subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "build>=1.2"], check=True)
build_env = os.environ.copy()
build_env["SOURCE_DATE_EPOCH"] = "1787760000"
subprocess.run(
    [sys.executable, "-m", "build", "--wheel", "--outdir", str(SDK_DIST), str(SOURCE_ROOT)],
    check=True, env=build_env
)
built = sorted(SDK_DIST.glob("kaggle_vllm-0.2.0.dev0-py3-none-any.whl"))
assert len(built) == 1, built
SDK_WHEEL = built[0]
print("SDK wheel:", SDK_WHEEL)
print("SDK SHA256:", sha256_file(SDK_WHEEL))

subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", str(SDK_WHEEL)], check=True)

import kaggle_vllm
assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION
print("SDK version:", kaggle_vllm.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-2acfc85e-3e76-87cf-1557-9be442cd9e80)
GPU 1: Tesla T4 (UUID: GPU-3ae79762-783d-7fc9-ff5c-9328b25bd1a0)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilat

/tmp/ipykernel_58/3255631628.py:39: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(destination)


* Creating isolated environment: venv+pip...
* Installing packages in isolated environment:
  - setuptools>=77
  - wheel
* Getting build dependencies for wheel...
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


running egg_info
creating src/kaggle_vllm.egg-info
writing src/kaggle_vllm.egg-info/PKG-INFO
writing dependency_links to src/kaggle_vllm.egg-info/dependency_links.txt
writing entry points to src/kaggle_vllm.egg-info/entry_points.txt
writing requirements to src/kaggle_vllm.egg-info/requires.txt
writing top-level names to src/kaggle_vllm.egg-info/top_level.txt
writing manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'
reading manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'
adding license file 'LICENSE'
writing manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'


* Building wheel...
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


running bdist_wheel
running build
running build_py
creating build/lib/kaggle_vllm
copying src/kaggle_vllm/download.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/server.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/__init__.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/bootstrap.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/cli.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/llm.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/dependencies.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/profiles.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/checksums.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/runtime.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/environment.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/sharding.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/doctor.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/installation.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/exceptions.py -> build/lib/kaggle_vllm
run

## 2. Strict native bootstrap, activation, native imports, and doctor

In [2]:
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-e2e-020-pr15")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

BOOTSTRAP = [
    "kaggle-vllm", "bootstrap", "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

subprocess.run(["kaggle-vllm", "fingerprint"], check=True)
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)
subprocess.run(BOOTSTRAP, check=True)
assert MANIFEST.is_file(), MANIFEST

from kaggle_vllm import activate_runtime
assert activate_runtime(MANIFEST)

import torch
import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

torch_after = f"{torch.__version__} {torch.version.cuda} {torch.__file__}\n"
assert torch_after == torch_before, (torch_before, torch_after)

doctor = subprocess.run(["kaggle-vllm", "doctor", "--strict", "--json"])
assert doctor.returncode == 0, doctor.returncode

print("Bootstrap/import/Torch/doctor: PASS")
print("vLLM:", getattr(vllm, "__version__", "unknown"), vllm.__file__)

{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/usr/local/nvidia/lib64/libcuda.so",
  "cmake_library_path": null,
  "driver_version": "580.159.04",


Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 233.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 262.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 337.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 304.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 237.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 196.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 315.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 242.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 281.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## 3. Raw NCCL two-GPU smoke test

In [3]:
NCCL_SCRIPT = SOURCE_ROOT / "scripts" / "nccl_smoke.py"
assert NCCL_SCRIPT.is_file(), NCCL_SCRIPT
subprocess.run([sys.executable, str(NCCL_SCRIPT)], check=True, env=os.environ.copy())
print("NCCL smoke: PASS")

rank=1 all_reduce=3.0
rank=0 all_reduce=3.0
NCCL all-reduce PASS
NCCL smoke: PASS


## 4. OPT-125M TP=1 and TP=2 in separate processes

In [4]:
SMOKE = r"""
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams
import sys
tp = int(sys.argv[1])
llm = KaggleLLM(
    model="facebook/opt-125m",
    tensor_parallel_size=tp,
    dtype="float16",
    max_model_len=512,
    gpu_memory_utilization=0.40,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)
out = llm.generate(
    [f"kaggle-vllm tensor parallel size {tp} validation:"],
    SamplingParams(temperature=0.0, max_tokens=32),
)
print(out[0].outputs[0].text)
"""

for tp in (1, 2):
    env = os.environ.copy()
    if tp == 1:
        env["CUDA_VISIBLE_DEVICES"] = "0"
    else:
        env.pop("CUDA_VISIBLE_DEVICES", None)
    print(f"=== OPT-125M TP={tp} ===")
    subprocess.run([sys.executable, "-c", SMOKE, str(tp)], check=True, env=env)
    time.sleep(2)

print("OPT-125M TP=1/TP=2: PASS")

=== OPT-125M TP=1 ===
INFO 08-30 13:51:23 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': 'facebook/opt-125m'}
INFO 08-30 13:51:43 [model.py:533] Resolved architecture: OPTForCausalLM
INFO 08-30 13:51:43 [model.py:1582] Using max model len 512
INFO 08-30 13:51:44 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-30 13:51:44 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 08-30 13:51:44 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-30 13:51:44 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-30 13:51:44 [vllm.py:985] Cudagraph is disabled under eager mode
INFO 08-30 13:51:4

[W830 13:52:05.907505732 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=315) INFO 08-30 13:52:06 [gpu_model_runner.py:4481] Starting to load model facebook/opt-125m...
(EngineCore pid=315) ERROR 08-30 13:52:06 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=315) INFO 08-30 13:52:06 [cuda.py:317] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=315) INFO 08-30 13:52:11 [weight_utils.py:574] Time spent downloading weights for facebook/opt-125m: 4.571946 seconds


Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.76it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.76it/s]
(EngineCore pid=315) 


(EngineCore pid=315) INFO 08-30 13:52:12 [default_loader.py:384] Loading weights took 0.27 seconds
(EngineCore pid=315) INFO 08-30 13:52:13 [gpu_model_runner.py:4566] Model loading took 0.24 GiB memory and 5.310491 seconds
(EngineCore pid=315) INFO 08-30 13:52:29 [gpu_worker.py:456] Available KV cache memory: 5.34 GiB
(EngineCore pid=315) INFO 08-30 13:52:29 [kv_cache_utils.py:1316] GPU KV cache size: 155,520 tokens
(EngineCore pid=315) INFO 08-30 13:52:29 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 303.75x
(EngineCore pid=315) INFO 08-30 13:52:29 [core.py:281] init engine (profile, create kv cache, warmup model) took 16.46 seconds
(EngineCore pid=315) INFO 08-30 13:52:30 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=315) WARNING 08-30 13:52:30 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=315) WARNING 08-30 13:52:30 [vllm.py:82

...



kaggle-vllm tensor parallel size 2 validation:

kaggle-vllm tensor parallel size 3 validation:
ERROR 08-30 13:52:34 [core_client.py:704] Engine core proc EngineCore died unexpectedly, shutting down client.
=== OPT-125M TP=2 ===
INFO 08-30 13:52:53 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': 'facebook/opt-125m'}
INFO 08-30 13:52:54 [model.py:533] Resolved architecture: OPTForCausalLM
INFO 08-30 13:52:54 [model.py:1582] Using max model len 512
INFO 08-30 13:52:54 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-30 13:52:54 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 08-30 13:52:54 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-30 13:52:54 [vllm.py:82

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.16it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.16it/s]
(Worker_TP0 pid=509) 


(Worker_TP0 pid=509) INFO 08-30 13:53:35 [default_loader.py:384] Loading weights took 0.32 seconds
(Worker_TP0 pid=509) INFO 08-30 13:53:36 [gpu_model_runner.py:4566] Model loading took 0.12 GiB memory and 1.010952 seconds
(Worker_TP0 pid=509) INFO 08-30 13:53:54 [gpu_worker.py:456] Available KV cache memory: 5.48 GiB
(EngineCore pid=485) INFO 08-30 13:53:54 [kv_cache_utils.py:1316] GPU KV cache size: 319,072 tokens
(EngineCore pid=485) INFO 08-30 13:53:54 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 623.19x
(EngineCore pid=485) INFO 08-30 13:53:55 [core.py:281] init engine (profile, create kv cache, warmup model) took 19.06 seconds
(EngineCore pid=485) INFO 08-30 13:53:57 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=485) WARNING 08-30 13:53:57 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=485) WARNING 08-30 13:53:57 [vllm.py:82

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.64s/it, est. speed input: 3.23 toks/s, output: 6.89 toks/s]


(EngineCore pid=485) INFO 08-30 13:54:02 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=485) INFO 08-30 13:54:02 [core.py:1224] Shutdown complete
(Worker_TP1 pid=510) INFO 08-30 13:54:02 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP1 pid=510) INFO 08-30 13:54:02 [multiproc_executor.py:854] WorkerProc shutting down.
(Worker_TP0 pid=509) INFO 08-30 13:54:02 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP0 pid=509) INFO 08-30 13:54:02 [multiproc_executor.py:854] WorkerProc shutting down.


kaggle-vllm tensor parallel size 2 validation:

kaggle-vllm tensor parallel size 2 validation:
ERROR 08-30 13:54:06 [core_client.py:704] Engine core proc EngineCore died unexpectedly, shutting down client.
OPT-125M TP=1/TP=2: PASS


## 5. Optional Qwen TP=2 sharded-state regression

The 0.1.1 executed notebook already contains full Qwen TP=2 sharded-state evidence.
Leave `RUN_QWEN = False` for a focused 0.2 SDK acceptance, or set it to `True` if you
want fresh end-to-end Qwen evidence in this run.

In [5]:
RUN_QWEN = False
QWEN_REPO = "waqasm86/kaggle-vllm-models"
QWEN_LOCAL = Path("/kaggle/working/kaggle-vllm-qwen-tp2")

if RUN_QWEN:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "huggingface_hub>=0.36"], check=True)
    from huggingface_hub import snapshot_download
    local_path = snapshot_download(repo_id=QWEN_REPO, local_dir=str(QWEN_LOCAL))
    print("Qwen snapshot:", local_path)
    subprocess.run(
        ["kaggle-vllm", "inspect-shards", str(QWEN_LOCAL), "--tensor-parallel-size", "2"],
        check=True,
    )
    qwen_code = r"""
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams
import sys
model = sys.argv[1]
llm = KaggleLLM(
    model=model,
    load_format="sharded_state",
    tensor_parallel_size=2,
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.70,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)
out = llm.generate(
    ["Explain tensor parallel inference in two concise sentences."],
    SamplingParams(temperature=0.0, max_tokens=64),
)
print(out[0].outputs[0].text)
"""
    subprocess.run([sys.executable, "-c", qwen_code, str(QWEN_LOCAL)], check=True, env=os.environ.copy())
    print("Qwen TP=2 sharded-state: PASS")
else:
    print("Qwen regression skipped; historical 0.1.1 evidence remains separate.")

Qwen regression skipped; historical 0.1.1 evidence remains separate.


## 6. Local OpenAI-compatible API on OPT-125M TP=2

In [6]:
import urllib.error

SERVER_LOG = Path("/kaggle/working/kaggle-vllm-020-openai-server.log")
served_name = "opt-125m-kaggle-t4x2"
server_cmd = [
    "vllm", "serve", "facebook/opt-125m",
    "--served-model-name", served_name,
    "--tensor-parallel-size", "2",
    "--dtype", "float16",
    "--max-model-len", "512",
    "--gpu-memory-utilization", "0.40",
    "--enforce-eager",
    "--disable-custom-all-reduce",
    "--host", "127.0.0.1",
    "--port", "8001",
]

with SERVER_LOG.open("w") as log:
    proc = subprocess.Popen(server_cmd, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

try:
    models_url = "http://127.0.0.1:8001/v1/models"
    ready = False
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM server exited early with code {proc.returncode}")
        try:
            with urllib.request.urlopen(models_url, timeout=2) as response:
                body = response.read().decode()
                print("GET /v1/models:", response.status, body[:500])
                if response.status == 200:
                    ready = True
                    break
        except Exception:
            time.sleep(1)
    assert ready, "OpenAI server did not become ready"

    payload = json.dumps({
        "model": served_name,
        "prompt": "CUDA tensor parallelism",
        "max_tokens": 24,
        "temperature": 0.0,
    }).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:8001/v1/completions",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as response:
        body = response.read().decode()
        print("POST /v1/completions:", response.status, body[:1000])
        assert response.status == 200

    print("OpenAI-compatible API: PASS")
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
    print("server return code:", proc.returncode)
    print("server log:", SERVER_LOG)

GET /v1/models: 200 {"object":"list","data":[{"id":"opt-125m-kaggle-t4x2","object":"model","created":1788098149,"owned_by":"vllm","root":"facebook/opt-125m","parent":null,"max_model_len":512,"permission":[{"id":"modelperm-a1435ec199b0b5de","object":"model_permission","created":1788098149,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}
POST /v1/completions: 200 {"id":"cmpl-a3a6126495c723e5","object":"text_completion","created":1788098149,"model":"opt-125m-kaggle-t4x2","choices":[{"index":0,"text":"\n\nThe CUDA tensor parallelism is a type of parallelism that is a type of parallelism that is","logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"prompt_logprobs":null,"prompt_token_ids":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":7,"total_tokens":31,"completion_tokens":24,"prompt_

## 7. Evidence manifest and final acceptance

In [7]:
EVIDENCE = Path("/kaggle/working/kaggle-vllm-020-acceptance-evidence.json")
evidence = {
    "status": "PASS",
    "sdk_version": EXPECTED_SDK_VERSION,
    "sdk_source_commit": EXPECTED_SOURCE_COMMIT,
    "sdk_wheel": SDK_WHEEL.name,
    "sdk_wheel_sha256": sha256_file(SDK_WHEEL),
    "runtime_manifest": str(MANIFEST),
    "runtime_manifest_sha256": sha256_file(MANIFEST),
    "torch_before": torch_before.strip(),
    "torch_after": torch_after.strip(),
    "qwen_regression_executed": RUN_QWEN,
}
EVIDENCE.write_text(json.dumps(evidence, indent=2) + "\n")
print(EVIDENCE.read_text())
print("Evidence SHA256:", sha256_file(EVIDENCE))
subprocess.run(["nvidia-smi"], check=False)
print("FINAL ACCEPTANCE: PASS")

{
  "status": "PASS",
  "sdk_version": "0.2.0.dev0",
  "sdk_source_commit": "7327b0b0c811a92a9c49421a4d302c18e251ab61",
  "sdk_wheel": "kaggle_vllm-0.2.0.dev0-py3-none-any.whl",
  "sdk_wheel_sha256": "a79f5d35a21a45838d9a7234e0764449ed6f3393a0c6fc9de6557f0485734ce4",
  "runtime_manifest": "/kaggle/working/kaggle-vllm-e2e-020-pr15/kaggle-vllm-runtime.json",
  "runtime_manifest_sha256": "c78ec5cd32fdda24a21a64f7fac108b0a4dbf0f14b171754563d352fa6b55835",
  "torch_before": "2.10.0+cu128 12.8 /usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_after": "2.10.0+cu128 12.8 /usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "qwen_regression_executed": false
}

Evidence SHA256: 4f0a61b1651070dedd8053e369a049ab7c50c7e98c1456ac71d6e6bac4ae18c4
Sun Aug 30 13:56:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-------------